# Lab 13.3: Explaining a Transaction Graph

Chapter 13 of *Explainable AI in Practice*. Part A checks two edge explainers against a planted ground truth on a small synthetic graph. Part B generates a **synthetic** payment network with planted money-laundering schemes, trains random forests and a graph convolutional network (GCN), and tests edge explanations against the schemes we planted. Nothing is downloaded; every number comes from data generated in this notebook. Run the cells in order (about 2 minutes on a laptop CPU).

In [1]:
import time, warnings
import numpy as np, pandas as pd, torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
warnings.filterwarnings("ignore")
torch.manual_seed(0); np.random.seed(0)

## Part A. Planted motifs: do the explainers find them?

A random tree-like graph of 300 nodes gets 40 planted 'houses' (five nodes, six edges). The only node feature is the node's number of links. A three-layer GCN learns to flag house nodes; we then explain correctly flagged nodes with single-edge occlusion and with a GNNExplainer-style learned mask, and count how many of each explainer's top six edges are house edges.

In [2]:
def ba_shapes(n_base=300, n_house=40, seed=0):
    rng = np.random.default_rng(seed)
    edges = [(0, 1), (1, 2), (2, 0)]; deg = np.zeros(n_base); deg[:3] = 2
    for v in range(3, n_base):
        u = rng.choice(v, p=deg[:v] / deg[:v].sum()); edges.append((v, u)); deg[v] += 1; deg[u] += 1
    lab = [0] * n_base
    for h in range(n_house):
        b = n_base + 5 * h
        edges += [(b, b + 1), (b + 1, b + 2), (b + 2, b + 3), (b + 3, b), (b, b + 4), (b + 1, b + 4)]
        lab += [1] * 5
        edges.append((b + 3, int(rng.integers(n_base))))
    n = n_base + 5 * n_house
    for _ in range(n // 10):
        a, c = rng.integers(n, size=2)
        if a != c: edges.append((int(a), int(c)))
    return n, np.array(edges), np.array(lab)
n, E, lab = ba_shapes()
def adj(w=None):
    s = np.r_[E[:, 0], E[:, 1], np.arange(n)]; d = np.r_[E[:, 1], E[:, 0], np.arange(n)]
    ww = torch.ones(len(E)) if w is None else w
    v = torch.cat([ww, ww, torch.ones(n)])
    deg = torch.zeros(n).index_add_(0, torch.tensor(s), v)
    return torch.sparse_coo_tensor(torch.tensor(np.vstack([s, d])), v / torch.sqrt(deg[s] * deg[d]), (n, n))
dg = np.bincount(E.ravel(), minlength=n)
X = torch.nn.functional.one_hot(torch.tensor(np.minimum(dg, 9)), 10).float()
class GCN3(torch.nn.Module):
    def __init__(s):
        super().__init__(); s.l = torch.nn.ModuleList([torch.nn.Linear(10, 20), torch.nn.Linear(20, 20), torch.nn.Linear(20, 20)]); s.o = torch.nn.Linear(60, 2)
    def forward(s, A):
        h = X; hs = []
        for l in s.l:
            h = torch.relu(torch.sparse.mm(A, l(h))); hs.append(h)
        return s.o(torch.cat(hs, 1))
g = GCN3(); opt = torch.optim.Adam(g.parameters(), lr=0.01)
y = torch.tensor(lab); A = adj(); cw = torch.tensor(len(lab) / (2.0 * np.bincount(lab)), dtype=torch.float32)
perm = np.random.default_rng(1).permutation(n); tr = torch.tensor(perm[:400]); te = torch.tensor(perm[400:])
for _ in range(1500):
    opt.zero_grad(); torch.nn.functional.cross_entropy(g(A)[tr], y[tr], weight=cw).backward(); opt.step()
with torch.no_grad():
    pr = g(A).argmax(1)
print("Checkpoint A1: held-out accuracy", round((pr[te] == y[te]).float().mean().item(), 2))

Checkpoint A1: held-out accuracy 0.95


In [3]:
logit = lambda out: (out[1] - out[0]).item()
def explain(v):
    nb = {v}
    for _ in range(3):
        nb |= {int(b) for a, b in E if a in nb} | {int(a) for a, b in E if b in nb}
    sub = [i for i, (a, b) in enumerate(E) if a in nb and b in nb]
    hb = 300 + 5 * ((v - 300) // 5)
    truth = np.array([E[i][0] in range(hb, hb + 5) and E[i][1] in range(hb, hb + 5) for i in sub])
    with torch.no_grad():
        l0 = logit(g(A)[v]); occ = []
        for i in sub:
            w = torch.ones(len(E)); w[i] = 0; occ.append(l0 - logit(g(adj(w))[v]))
    m = torch.nn.Parameter(torch.full((len(sub),), 2.0)); o2 = torch.optim.Adam([m], lr=0.05)
    for _ in range(300):
        o2.zero_grad()
        w = torch.zeros(len(E)).index_put((torch.tensor(sub),), torch.sigmoid(m))
        sm = torch.sigmoid(m)
        ent = -(sm * torch.log(sm + 1e-8) + (1 - sm) * torch.log(1 - sm + 1e-8)).mean()
        (-torch.log_softmax(g(adj(w))[v], 0)[1] + 0.05 * sm.sum() + 0.5 * ent).backward(); o2.step()
    k = int(truth.sum())
    hit = lambda sc: truth[np.argsort(-np.asarray(sc))[:k]].mean()
    return len(sub), k, hit(occ), hit(torch.sigmoid(m).detach().numpy()), truth.mean()
res = [explain(int(v)) for v in te if lab[v] == 1 and pr[v] == 1]
R = pd.DataFrame(res, columns=["edges", "motif_edges", "occlusion", "learned_mask", "random"])
print("Checkpoint A2: mean share of top edges that are motif edges, over", len(R), "nodes")
print(R[["occlusion", "learned_mask", "random"]].mean().round(2).to_string())

Checkpoint A2: mean share of top edges that are motif edges, over 43 nodes
occlusion       0.76
learned_mask    0.65
random          0.55


## Part B. A synthetic payment network with planted laundering schemes

The data are **synthetic**, generated below; they are not real payments and not a sample of any bank's data. Each of 30 time windows has about 1,000 accounts (15 percent businesses) making ordinary payments: purchases, transfers between people, salaries and supplier payments, with a few large legitimate transfers. Into each window we plant 10 laundering schemes of three shapes (typologies) used in anti-money-laundering simulators: **fan-out** (one hub pays 4 to 7 mules), **fan-in** (4 to 7 mules pay one hub) and **scatter-gather** (a hub pays mules who pass the money on to a second hub). Most mules are existing customers recruited into the scheme, so they keep their ordinary activity; 30 percent of schemes keep amounts just under 10,000. Labels: 1 = account in a scheme (hub or mule). Because we planted the schemes, we know which payments are the true reason for each label, which real data never tell you.

In [4]:
BAND = (8000, 10000)

def make_window(rng, w, n=1000, n_pat=10, p_band=0.3, p_recruit=0.7):
    biz = rng.random(n) < 0.15; age = rng.lognormal(6.0, 1.0, n); pop = rng.pareto(1.5, n) + 1
    bz = np.where(biz)[0]; pr = np.where(~biz)[0]
    S, D, A, pat = [], [], [], []
    def pay(s, d, a, k=-1):
        S.append(int(s)); D.append(int(d)); A.append(float(a)); pat.append(k)
    def person_activity(p, lam=3):
        for _ in range(rng.poisson(lam)):
            r = rng.random()
            if r < 0.72: pay(p, rng.choice(bz, p=pop[bz] / pop[bz].sum()), rng.lognormal(4.0, 1.0))
            elif r < 0.96: pay(p, rng.choice(pr), rng.lognormal(4.5, 1.2))
            else: pay(p, rng.choice(pr), rng.uniform(*BAND))            # a large legitimate transfer
    for p in pr: person_activity(p)
    for b in bz:                                                         # salaries and suppliers
        for _ in range(rng.poisson(4)): pay(b, rng.choice(pr), rng.lognormal(7.5, 0.3))
        for _ in range(rng.poisson(2)): pay(b, rng.choice(bz, p=pop[bz] / pop[bz].sum()), rng.lognormal(7.0, 1.1))
    ages = list(age); bizl = list(biz); roles = {}
    def account(recruit, business=False):
        if rng.random() < recruit:                                       # recruit an existing customer
            c = int(rng.choice(bz if business else pr))
            if c not in roles: return c
        ages.append(rng.lognormal(5.0, 1.2)); bizl.append(business); v = len(ages) - 1
        if not business: person_activity(v, 2)                          # new accounts look ordinary too
        return v
    for k in range(n_pat):
        typ = ["fan-out", "fan-in", "scatter-gather"][k % 3]; m = int(rng.integers(4, 8))
        under = rng.random() < p_band
        amt = (lambda: rng.uniform(*BAND)) if under else (lambda: rng.lognormal(7.6, 0.5))
        shell = rng.random() < 0.5
        hub = account(0.3, shell); roles[hub] = 1
        mules = [account(p_recruit) for _ in range(m)]
        for u in mules: roles.setdefault(u, 2)
        pid = w * 100 + k
        if typ == "fan-out":
            for u in mules: pay(hub, u, amt(), pid)
        elif typ == "fan-in":
            for u in mules: pay(u, hub, amt(), pid)
        else:
            sink = account(0.3, shell); roles[sink] = 1
            for u in mules:
                a = amt(); pay(hub, u, a, pid); pay(u, sink, a * rng.uniform(0.9, 0.98), pid)
    rl = np.zeros(len(ages), int)
    for u, r in roles.items(): rl[u] = r
    acc = pd.DataFrame(dict(window=w, local=np.arange(len(ages)), age=ages, biz=bizl, role=rl))
    return acc, pd.DataFrame(dict(window=w, src=S, dst=D, amount=A, pattern=pat))


In [5]:
FEATURES = ["payments in", "payments out", "log amount in", "log amount out", "share just under 10,000",
            "pass-through ratio", "spread of amounts out", "log account age", "business"]
def local_features(acc, tx):
    N = len(acc); ix = range(N)
    og, ig = tx.groupby("src").amount, tx.groupby("dst").amount
    nin, nout = ig.size().reindex(ix, fill_value=0).to_numpy(), og.size().reindex(ix, fill_value=0).to_numpy()
    sin, sout = ig.sum().reindex(ix, fill_value=0).to_numpy(), og.sum().reindex(ix, fill_value=0).to_numpy()
    b = tx[tx.amount.between(*BAND)]
    nb = (b.groupby("src").size().reindex(ix, fill_value=0) + b.groupby("dst").size().reindex(ix, fill_value=0)).to_numpy()
    cv = (og.std() / og.mean()).reindex(ix).fillna(1.0).to_numpy()
    return np.c_[nin, nout, np.log1p(sin), np.log1p(sout), nb / np.maximum(nin + nout, 1),
                 np.minimum(sin, sout) / np.maximum(np.maximum(sin, sout), 1), cv, np.log(acc.age), acc.biz]
rng = np.random.default_rng(0); ACC, TX, FL, off = [], [], [], 0
for w in range(1, 31):
    a, t = make_window(rng, w); FL.append(local_features(a, t))
    a["node"] = a.local + off; t["s"] = t.src + off; t["d"] = t.dst + off
    off += len(a); ACC.append(a); TX.append(t)
acc = pd.concat(ACC, ignore_index=True); tx = pd.concat(TX, ignore_index=True); F = np.vstack(FL).astype(np.float32)
y = (acc.role > 0).astype(int).to_numpy(); role = acc.role.to_numpy(); win = acc.window.to_numpy(); N = len(acc)
e = pd.DataFrame(dict(a=np.minimum(tx.s, tx.d), b=np.maximum(tx.s, tx.d), pattern=tx.pattern))
E = e.groupby(["a", "b"]).pattern.max().reset_index()                    # one undirected edge per pair of accounts
src, dst, epat = E.a.to_numpy(), E.b.to_numpy(), E.pattern.to_numpy()
print("Checkpoint B1: accounts", N, "payments", len(tx), "edges", len(E), "laundering accounts", y.sum(),
      f"(hubs {(role == 1).sum()}, mules {(role == 2).sum()})", "scheme payments", (tx.pattern >= 0).sum())


Checkpoint B1: accounts 30791 payments 107488 edges 104582 laundering accounts 2035 (hubs 390, mules 1645) scheme payments 2148


Windows 1-20 are for training and 21-30 for testing. The first forest sees only each account's own nine features; the second also sees summaries of its neighbors (the mean and maximum of each feature over the accounts it paid or was paid by), the way some public datasets put the graph inside the table.

In [6]:
tr, te = win <= 20, win >= 21
nbr = pd.DataFrame(dict(u=np.r_[src, dst], v=np.r_[dst, src]))
G = pd.DataFrame(F).iloc[nbr.v].set_index(nbr.u.to_numpy()).groupby(level=0)
FA = np.hstack([F, G.mean().reindex(range(N)).fillna(0).to_numpy(), G.max().reindex(range(N)).fillna(0).to_numpy()])
for name, X in (("own features", F), ("own + neighbor summaries", FA)):
    rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=0).fit(X[tr], y[tr])
    p = rf.predict(X[te])
    print(f"Checkpoint B2: random forest, {name}: P {precision_score(y[te], p):.3f} R {recall_score(y[te], p):.3f} F1 {f1_score(y[te], p):.3f}",
          f"| recall hubs {p[role[te] == 1].mean():.2f}, mules {p[role[te] == 2].mean():.2f}")


Checkpoint B2: random forest, own features: P 0.689 R 0.495 F1 0.576 | recall hubs 0.77, mules 0.43


Checkpoint B2: random forest, own + neighbor summaries: P 0.882 R 0.544 F1 0.673 | recall hubs 0.69, mules 0.51


The GCN sees only the nine own features; everything it knows about neighbors comes through the edges, by message passing over two layers.

In [7]:
def norm_adj(s, d, nn, w=None):
    s2 = np.concatenate([s, d, np.arange(nn)]); d2 = np.concatenate([d, s, np.arange(nn)])
    w = torch.ones(len(s)) if w is None else w
    v = torch.cat([w, w, torch.ones(nn)])
    deg = torch.zeros(nn).index_add_(0, torch.tensor(s2), v)
    return torch.sparse_coo_tensor(torch.tensor(np.vstack([s2, d2])), v / torch.sqrt(deg[s2] * deg[d2]), (nn, nn)).coalesce()
mu, sd = F[tr].mean(0), F[tr].std(0) + 1e-6
Xt = torch.tensor((F - mu) / sd, dtype=torch.float32)
class GCN(torch.nn.Module):
    def __init__(s, h=64):
        super().__init__(); s.l1 = torch.nn.Linear(9, h); s.l2 = torch.nn.Linear(h, h); s.o = torch.nn.Linear(9 + h, 2)
    def forward(s, A, X):
        h = torch.relu(s.l2(torch.sparse.mm(A, torch.relu(s.l1(torch.sparse.mm(A, X))))))
        return s.o(torch.cat([X, h], 1))
torch.manual_seed(0); t0 = time.time()
gcn = GCN(); opt = torch.optim.Adam(gcn.parameters(), lr=0.01, weight_decay=1e-4)
AE = norm_adj(src, dst, N); yt = torch.tensor(y); tri = torch.tensor(np.where(tr)[0])
cw = torch.tensor([1.0, (1 - y[tr].mean()) / y[tr].mean()], dtype=torch.float32) ** 0.5
for ep in range(300):
    opt.zero_grad(); torch.nn.functional.cross_entropy(gcn(AE, Xt)[tri], yt[tri], weight=cw).backward(); opt.step()
with torch.no_grad():
    P = torch.softmax(gcn(AE, Xt), 1)[:, 1].numpy()
    P0 = torch.softmax(gcn(norm_adj(np.array([], int), np.array([], int), N), Xt), 1)[:, 1].numpy()   # every edge removed
pt = (P > 0.5).astype(int)
print(f"trained in {time.time() - t0:.0f} s")
print(f"Checkpoint B3: GCN P {precision_score(y[te], pt[te]):.3f} R {recall_score(y[te], pt[te]):.3f} F1 {f1_score(y[te], pt[te]):.3f}",
      f"| recall hubs {pt[te & (role == 1)].mean():.2f}, mules {pt[te & (role == 2)].mean():.2f}")
for r, nm in ((1, "hubs"), (2, "mules")):
    fl = te & (role == r) & (pt == 1)
    print(f"Checkpoint B4: {nm}: {fl.sum()} correctly flagged; still flagged with every edge removed: {(P0[fl] > 0.5).mean():.2f}")


trained in 10 s
Checkpoint B3: GCN P 0.650 R 0.709 F1 0.678 | recall hubs 0.73, mules 0.70
Checkpoint B4: hubs: 95 correctly flagged; still flagged with every edge removed: 0.64
Checkpoint B4: mules: 382 correctly flagged; still flagged with every edge removed: 0.92


## Part B, continued. Edge explanations against the planted schemes

For a sample of correctly flagged test accounts (up to 40 hubs and 80 mules), we remove the edges within two steps of the account one at a time (single-edge occlusion) and rank them by how much the GCN's probability falls. With k = the number of true scheme edges among the candidates, we count how many of the top-k edges are scheme edges, against the share a random pick would get. Then we remove all the scheme's edges together, and finally every edge around the account.

In [8]:
def explain(v):
    nodes = np.where(win == win[v])[0]; loc = pd.Series(np.arange(len(nodes)), index=nodes)
    em = np.isin(src, nodes); s_, d_, pp, lv = loc.loc[src[em]].to_numpy(), loc.loc[dst[em]].to_numpy(), epat[em], loc[v]
    inc = np.where((s_ == lv) | (d_ == lv))[0]; hop1 = list(set(s_[inc]) | set(d_[inc]))
    cand = np.where(np.isin(s_, hop1) | np.isin(d_, hop1))[0]
    truth = np.isin(pp[cand], [q for q in set(pp[inc]) if q >= 0])          # edges of the account's own scheme
    Xs = Xt[nodes]
    def prob(cut):
        w = torch.ones(len(s_)); w[cut] = 0
        with torch.no_grad():
            return torch.softmax(gcn(norm_adj(s_, d_, len(nodes), w), Xs)[lv], 0)[1].item()
    base = prob([]); drops = np.array([base - prob([c]) for c in cand]); k = int(truth.sum())
    return dict(account=int(v), role="hub" if role[v] == 1 else "mule", edges=len(cand), scheme_edges=k,
                top_k_hit=truth[np.argsort(-drops)[:k]].mean(), random=truth.mean(), p=base,
                best_single=drops.max(), sum_single_scheme=drops[truth].sum(),
                p_scheme_cut=prob(cand[truth]), p_all_cut=prob(cand))
rng2 = np.random.default_rng(0); t0 = time.time()
pick = [v for r, n_ in ((1, 40), (2, 80)) for v in rng2.permutation(np.where(te & (role == r) & (pt == 1))[0])[:n_]]
R = pd.DataFrame([explain(v) for v in pick])
print(f"explained {len(R)} accounts in {time.time() - t0:.0f} s")
S = R.groupby("role").agg(n=("account", "size"), top_k_hit=("top_k_hit", "mean"), random=("random", "mean"),
                          reversed_scheme_cut=("p_scheme_cut", lambda x: (x < 0.5).mean()),
                          reversed_all_cut=("p_all_cut", lambda x: (x < 0.5).mean()))
print("Checkpoint B5: share of top-k edges that are scheme edges, and flags reversed by cutting edges")
print(S.round(2).to_string())


explained 120 accounts in 27 s
Checkpoint B5: share of top-k edges that are scheme edges, and flags reversed by cutting edges
       n  top_k_hit  random  reversed_scheme_cut  reversed_all_cut
role                                                              
hub   40       0.55    0.16                 0.60              0.28
mule  80       0.71    0.17                 0.61              0.08


In [9]:
M = R[R.role == "mule"]
gap = (M.sum_single_scheme - (M.p - M.p_scheme_cut)).abs()
print(f"Checkpoint B6: mules: sum of single scheme-edge drops vs joint drop, median absolute gap {gap.median():.2f}, largest {gap.max():.2f}")
ex = M[(M.p_scheme_cut < 0.5) & (M.p_all_cut > 0.5)].sort_values(["scheme_edges", "p"], ascending=[True, False]).head(1)
print("Example:", ex[["account", "edges", "scheme_edges", "p", "p_scheme_cut", "p_all_cut"]].round(2).to_dict("records"))


Checkpoint B6: mules: sum of single scheme-edge drops vs joint drop, median absolute gap 0.24, largest 1.35
Example: [{'account': 23204, 'edges': 83, 'scheme_edges': 4, 'p': 0.82, 'p_scheme_cut': 0.03, 'p_all_cut': 1.0}]


## Your turn

Pick one flagged hub from `R`, list its five most important edges by single-edge occlusion, then cut all five together. Does the joint drop equal the sum of the single drops? Then write the explanation you would give an investigator: which payments, which shape, and what the model's own features contributed.